# 🎵 YuE2-3B Studio — Google Colab Edition (12 GB VRAM Optimized)
Интерфейс для генерации музыки, MIDI и лупов с выводом в интернет через **ngrok**.

**Порядок запуска:**
1. В меню сверху выберите: `Среда выполнения (Runtime)` -> `Сменить среду выполнения` -> **T4 GPU**.
2. Выполните **Ячейку 1** (установка библиотек и клонирование репозитория). В конце сеанс автоматически перезагрузится — это нормально!
3. Введите токены в **Ячейке 2** и запустите её.
4. Запустите **Ячейку 3** — получите ссылку ngrok и откройте студию в браузере.

In [ ]:
# 1. Клонирование репозитория и установка библиотек
!nvidia-smi

%cd /content
!rm -rf YuE2-3B-Studio
!git clone https://github.com/cristie-music/YuE2-3B-Studio.git
%cd /content/YuE2-3B-Studio

# Установка ядра моделей и утилит
!pip install --no-warn-conflicts "huggingface-hub==0.36.2" "tokenizers==0.21.4" "transformers==4.49.0"
!pip install --no-warn-conflicts pyngrok soundfile music21 "diffusers==0.32.2" "accelerate==1.13.0" "peft==0.14.0" bitsandbytes onnxruntime-gpu
!pip install ./yue2_infer-0.1.5-py3-none-any.whl --no-deps

print("\n[OK] Пакеты установлены. Перезапуск ядра сессии для применения системных модулей...")
import os
os._exit(0)

In [ ]:
#@title ⚙️ 2. Параметры доступа (HF Token & Ngrok)
import os

HF_TOKEN = "hf_YOUR_HF_TOKEN_HERE" #@param {type:"string"}
NGROK_AUTHTOKEN = "YOUR_NGROK_AUTHTOKEN_HERE" #@param {type:"string"}

os.environ["HF_TOKEN"] = HF_TOKEN.strip()
os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTHTOKEN.strip()
print("Токены сохранены в переменных окружения.")

In [ ]:
# 3. Запуск туннеля ngrok и сервера YuE2 Studio
import os
import time
from pyngrok import ngrok

%cd /content/YuE2-3B-Studio

ngrok_token = os.environ.get("NGROK_AUTHTOKEN", "").strip()
if not ngrok_token or ngrok_token == "YOUR_NGROK_AUTHTOKEN_HERE":
    raise ValueError("Пожалуйста, укажите NGROK_AUTHTOKEN в ячейке №2!")

ngrok.set_auth_token(ngrok_token)

# Закрываем старые туннели, если они оставались открыты
ngrok.kill()

# Открываем порт 7860 через ngrok
ngrok_tunnel = ngrok.connect(7860)
public_url = ngrok_tunnel.public_url

print("=" * 65)
print(f"🚀 ВАША ВЕБ-СТУДИЯ ДОСТУПНА ПО ССЫЛКЕ:")
print(f"👉 {public_url}")
print("=" * 65)

server_script = "web_server_colab.py" if os.path.exists("web_server_colab.py") else "web_server.py"
!python {server_script}